In [13]:
import os
from dataclasses import dataclass
from typing import Optional, Tuple
from scipy.io import wavfile

import numpy as np

In [14]:

@dataclass
class MFCCConfig:
    sample_rate: int = 16000
    frame_length_ms: float = 30.0
    frame_overlap_ms: float = 10.0
    n_fft: int = 512
    n_mels: int = 23
    n_mfcc_without_energy: int = 12
    fmin: float = 0.0
    fmax: Optional[float] = None
    top_db_trim: int = 30
    center: bool = False
    lifter: int = 0
    eps: float = 1e-10


class CatMFCCExtractor:
    """
    MFCC extractor matching the paper's feature design as closely as possible:
      - 23 mel filterbank log-energies
      - DCT -> keep 12 coefficients
      - append log frame energy -> 13 base dimensions
      - append delta, delta-delta, delta-delta-delta

    Final output shape:
      (num_frames, 52)
    """

    def __init__(self, config: MFCCConfig):
        self.cfg = config

        if self.cfg.fmax is None:
            self.cfg.fmax = self.cfg.sample_rate / 2.0

        self.win_length = int(round(self.cfg.sample_rate * self.cfg.frame_length_ms / 1000.0))
        overlap = int(round(self.cfg.sample_rate * self.cfg.frame_overlap_ms / 1000.0))
        self.hop_length = self.win_length - overlap

        if self.hop_length <= 0:
            raise ValueError("Hop length must be positive. Check frame length and overlap.")

    def load_audio(path, target_sr=16000):
        sr, y = wavfile.read(path)

        if y.dtype == np.int16:
            y = y.astype(np.float32) / 32768.0
        elif y.dtype == np.int32:
            y = y.astype(np.float32) / 2147483648.0
        elif y.dtype == np.uint8:
            y = (y.astype(np.float32) - 128.0) / 128.0
        else:
            y = y.astype(np.float32)

        if y.ndim == 2:
            y = np.mean(y, axis=1)

        return y, sr

    def trim_silence(self, y: np.ndarray) -> np.ndarray:
        """
        Approximate silence removal. The paper uses a statistical-model-based silence
        elimination method; this is a practical substitute. :contentReference[oaicite:1]{index=1}
        """
        yt, _ = librosa.effects.trim(y, top_db=self.cfg.top_db_trim)
        if yt.size == 0:
            return y
        return yt.astype(np.float32)

    def pre_emphasis(self, y: np.ndarray, coef: float = 0.97) -> np.ndarray:
        """
        Optional pre-emphasis. Not explicitly stated in the paper, so leave unused
        by default unless you want a more speech-style pipeline.
        """
        if y.size < 2:
            return y
        return np.append(y[0], y[1:] - coef * y[:-1]).astype(np.float32)

    def _power_spectrogram(self, y: np.ndarray) -> np.ndarray:
        """
        Compute power spectrogram with:
          - Hamming window
          - win_length from 30 ms
          - hop_length from 30 ms frame / 10 ms overlap
          - FFT size 512
        Returns:
          S_power: shape (1 + n_fft/2, num_frames)
        """
        stft = librosa.stft(
            y,
            n_fft=self.cfg.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window="hamming",
            center=self.cfg.center,
            pad_mode="constant",
        )
        return np.abs(stft) ** 2

    def _log_mel_energies(self, power_spec: np.ndarray) -> np.ndarray:
        """
        Apply a 23-band mel filterbank and take log energies.
        Returns:
          log_mel: shape (n_mels, num_frames)
        """
        mel_basis = librosa.filters.mel(
            sr=self.cfg.sample_rate,
            n_fft=self.cfg.n_fft,
            n_mels=self.cfg.n_mels,
            fmin=self.cfg.fmin,
            fmax=self.cfg.fmax,
            htk=False,
            norm="slaney",
        )

        mel_energies = mel_basis @ power_spec
        mel_energies = np.maximum(mel_energies, self.cfg.eps)
        log_mel = np.log(mel_energies)
        return log_mel

    def _frame_log_energy(self, y: np.ndarray) -> np.ndarray:
        """
        Compute frame-wise log energy aligned with the STFT frames.

        Returns:
          log_energy: shape (num_frames,)
        """
        frames = librosa.util.frame(
            y,
            frame_length=self.win_length,
            hop_length=self.hop_length
        )

        # Apply the same Hamming window used in the STFT
        window = np.hamming(self.win_length).astype(np.float32)[:, None]
        windowed = frames * window

        energy = np.sum(windowed ** 2, axis=0)
        energy = np.maximum(energy, self.cfg.eps)
        return np.log(energy)

    def _mfcc_base(self, log_mel: np.ndarray, log_energy: np.ndarray) -> np.ndarray:
        """
        DCT of log mel energies -> keep 12 coefficients.
        Then append log frame energy to make 13 base features.

        Returns:
          base: shape (13, num_frames)
        """
        mfcc12 = librosa.feature.mfcc(
            S=log_mel,
            n_mfcc=self.cfg.n_mfcc_without_energy,
            dct_type=2,
            norm="ortho",
            lifter=self.cfg.lifter,
        )

        if mfcc12.shape[1] != log_energy.shape[0]:
            raise ValueError(
                f"Frame mismatch: mfcc has {mfcc12.shape[1]} frames, "
                f"log energy has {log_energy.shape[0]} frames."
            )

        base = np.vstack([mfcc12, log_energy[None, :]])
        return base

    def _derivatives(self, feat: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Compute 1st, 2nd, and 3rd derivatives.
        feat shape: (num_features, num_frames)
        """
        delta1 = librosa.feature.delta(feat, order=1, mode="nearest")
        delta2 = librosa.feature.delta(feat, order=2, mode="nearest")
        delta3 = librosa.feature.delta(feat, order=3, mode="nearest")
        return delta1, delta2, delta3

    def extract_sequence(
        self,
        path: Optional[str] = None,
        y: Optional[np.ndarray] = None,
        trim_silence: bool = True,
        use_pre_emphasis: bool = False,
    ) -> np.ndarray:
        """
        Extract frame-level MFCC sequence.

        Input:
          either `path` or `y` must be provided.

        Returns:
          features: shape (num_frames, 52)
        """
        if path is None and y is None:
            raise ValueError("Provide either `path` or `y`.")
        if path is not None and y is not None:
            raise ValueError("Provide only one of `path` or `y`, not both.")

        if path is not None:
            y = self.load_audio(path)

        if trim_silence:
            y = self.trim_silence(y)

        if use_pre_emphasis:
            y = self.pre_emphasis(y)

        if y.size < self.win_length:
            # Pad short clips so at least one frame can be formed
            pad = self.win_length - y.size
            y = np.pad(y, (0, pad), mode="constant")

        power_spec = self._power_spectrogram(y)
        log_mel = self._log_mel_energies(power_spec)
        log_energy = self._frame_log_energy(y)
        base = self._mfcc_base(log_mel, log_energy)
        delta1, delta2, delta3 = self._derivatives(base)

        full = np.vstack([base, delta1, delta2, delta3])  # (52, T)
        return full.T.astype(np.float32)  # (T, 52)

    def extract_clip_summary(
        self,
        path: Optional[str] = None,
        y: Optional[np.ndarray] = None,
        trim_silence: bool = True,
        use_pre_emphasis: bool = False,
    ) -> np.ndarray:
        """
        Optional helper:
        Convert the frame-level sequence into a fixed-length clip vector.

        Summary = [mean, std] over time
        Output dimension: 104
        """
        seq = self.extract_sequence(
            path=path,
            y=y,
            trim_silence=trim_silence,
            use_pre_emphasis=use_pre_emphasis,
        )

        mean = np.mean(seq, axis=0)
        std = np.std(seq, axis=0)
        return np.concatenate([mean, std]).astype(np.float32)


def save_feature_sequence(features: np.ndarray, out_path: str) -> None:
    """
    Save frame-level features to .npy file.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    np.save(out_path, features)

In [ ]:


if __name__ == "__main__":
    config = MFCCConfig(
        sample_rate=16000,
        frame_length_ms=30.0,
        frame_overlap_ms=10.0,
        n_fft=512,
        n_mels=23,
        n_mfcc_without_energy=12,
    )

    extractor = CatMFCCExtractor(config)

    audio_path = ""
    seq = extractor.extract_sequence(audio_path)

    print("Frame-level feature shape:", seq.shape)
    # Expected: (num_frames, 52)

    summary = extractor.extract_clip_summary(audio_path)
    print("Clip summary shape:", summary.shape)
    # Expected: (104,)

TypeError: expected str, bytes or os.PathLike object, not CatMFCCExtractor